
# 04 NPS Gold-Standard Expansion and Evaluation Splits

Notebook purpose: Based on previous notebooks, expand the human-reviewed gold-standard dataset from 120 to 300 alerts because of low numbers in some classes. Further, target underrepresented traveler-centered classes while preserving park diversity. Evaluation: Prevent exact duplicate text from leaking across evaluation partitions, create the held-out-park split as the primary generalization setting, and create a mixed-park split as a secondary, easier comparison.


## Design decisions based on professor feedback and previous notebooks

The classification task will be the main project.
The held-out-park result will be reported first and emphasized in the final paper.
The mixed-park split will also be reported but treated as an easier setting.
Additional records are suggested for annotation. LLM summarization remains outside this notebook and may be attempted only after classification is complete.



## 1. Upload required files

The notebook needs:

- `nps_alerts_enriched_20260623T190917Z.csv`
- `nps_taxonomy_annotation_sample_COMPLETED.xlsx`



In [ ]:

from pathlib import Path
import re
import json
import math
import random
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

SEARCH_DIRS = [Path("/content"), Path("/mnt/data"), Path(".")]

def find_file(filename):
    for directory in SEARCH_DIRS:
        candidate = directory / filename
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find {filename}. Upload it to /content/ or update SEARCH_DIRS."
    )

ALERTS_PATH = find_file("nps_alerts_enriched_20260623T190917Z.csv")
ANNOTATIONS_PATH = find_file("nps_taxonomy_annotation_sample_COMPLETED.xlsx")

print("Alerts file:", ALERTS_PATH)
print("Completed annotations:", ANNOTATIONS_PATH)


Alerts file: /content/nps_alerts_enriched_20260623T190917Z.csv
Completed annotations: /content/nps_taxonomy_annotation_sample_COMPLETED.xlsx


## 2. Load and validate the collected alerts and completed pilot annotations

In [ ]:

alerts_df = pd.read_csv(ALERTS_PATH)
gold_120_df = pd.read_excel(
    ANNOTATIONS_PATH,
    sheet_name="nps_taxonomy_annotation_sample_"
)

print("Full alert collection:", alerts_df.shape)
print("Completed pilot annotations:", gold_120_df.shape)

required_alert_columns = {
    "id", "parkCode", "title", "description", "category", "normalized_text"
}
required_gold_columns = {
    "id", "parkCode", "primary_label", "impact_label",
    "review_status", "duplicate_group_id"
}

missing_alert = required_alert_columns - set(alerts_df.columns)
missing_gold = required_gold_columns - set(gold_120_df.columns)

if missing_alert:
    raise ValueError(f"Missing alert columns: {sorted(missing_alert)}")
if missing_gold:
    raise ValueError(f"Missing annotation columns: {sorted(missing_gold)}")

gold_120_df["primary_label"] = gold_120_df["primary_label"].astype(str).str.strip()
gold_120_df["impact_label"] = gold_120_df["impact_label"].astype(str).str.strip()

print("\nCurrent primary-label counts:")
display(gold_120_df["primary_label"].value_counts().to_frame("count"))

print("\nCurrent impact-label counts:")
display(gold_120_df["impact_label"].value_counts().to_frame("count"))


Full alert collection: (627, 20)
Completed pilot annotations: (120, 20)

Current primary-label counts:


,count
primary_label,
road_parking_transportation,32
trail_or_area_access,27
facility_water_campground_service,26
weather_fire_environmental_hazard,20
wildlife_hazard_or_restriction,8
construction_maintenance_general,7



Current impact-label counts:


,count
impact_label,
requires_preparation,62
trip_changing,37
informational_minimal_impact,21



## 3. Freeze the final taxonomy

The pilot does support six primary traveler-centered classes.

The target expansion is approximately 300 total labeled alerts, with about 50 examples per class. This is merely a sampling goal rather than a requirement that the natural population be perfectly balanced.


In [ ]:

PRIMARY_LABELS = [
    "trail_or_area_access",
    "road_parking_transportation",
    "weather_fire_environmental_hazard",
    "wildlife_hazard_or_restriction",
    "facility_water_campground_service",
    "construction_maintenance_general",
]

IMPACT_LABELS = [
    "trip_changing",
    "requires_preparation",
    "informational_minimal_impact",
]

TARGET_TOTAL = 300
TARGET_PER_CLASS = 50

current_counts = (
    gold_120_df["primary_label"]
    .value_counts()
    .reindex(PRIMARY_LABELS, fill_value=0)
)

target_counts = pd.Series(TARGET_PER_CLASS, index=PRIMARY_LABELS)
target_deficits = (target_counts - current_counts).clip(lower=0)

planning_table = pd.DataFrame({
    "current_count": current_counts,
    "target_count": target_counts,
    "suggested_additional": target_deficits,
})

print("Planned additional annotations:", int(target_deficits.sum()))
display(planning_table)


Planned additional annotations: 180


,current_count,target_count,suggested_additional
trail_or_area_access,27,50,23
road_parking_transportation,32,50,18
weather_fire_environmental_hazard,20,50,30
wildlife_hazard_or_restriction,8,50,42
facility_water_campground_service,26,50,24
construction_maintenance_general,7,50,43



## 4. Prepare unlabeled sample

Remove the obvious test alert, removes alerts already in the 120-record gold set, removes empty text, retains one representative per exact normalized-text group for candidate selection, and preserves duplicate-group information for later leakage control.


In [ ]:

def normalize_text(text):
    text = "" if pd.isna(text) else str(text)
    return re.sub(r"\s+", " ", text.lower()).strip()

alerts_df["normalized_text"] = alerts_df["normalized_text"].fillna(
    (alerts_df["title"].fillna("") + " " + alerts_df["description"].fillna(""))
    .map(normalize_text)
)

alerts_df["combined_text_raw"] = (
    alerts_df["title"].fillna("").astype(str).str.strip()
    + " "
    + alerts_df["description"].fillna("").astype(str).str.strip()
).str.strip()

test_mask = (
    alerts_df["title"].fillna("").str.strip().str.lower().eq("test alert")
    | alerts_df["combined_text_raw"].str.lower().str.contains(
        r"\bthis is (a )?test alert\b", regex=True
    )
)

already_labeled_ids = set(gold_120_df["id"].astype(str))

candidate_pool = alerts_df.loc[
    ~test_mask
    & ~alerts_df["id"].astype(str).isin(already_labeled_ids)
    & alerts_df["combined_text_raw"].str.strip().ne("")
].copy()

# Use normalized text as a cross-park duplicate key.
candidate_pool["duplicate_text_key"] = candidate_pool["normalized_text"]
candidate_pool["duplicate_text_group_size_global"] = (
    candidate_pool.groupby("duplicate_text_key")["id"].transform("size")
)

# One representative per exact text group for the expansion worksheet.
candidate_unique = (
    candidate_pool
    .sort_values(["duplicate_text_group_size_global", "parkCode", "id"])
    .drop_duplicates("duplicate_text_key", keep="first")
    .copy()
)

print("Unlabeled candidate records:", len(candidate_pool))
print("Unique exact-text candidates:", len(candidate_unique))
print("Parks in candidate pool:", candidate_unique["parkCode"].nunique())


Unlabeled candidate records: 506
Unique exact-text candidates: 481
Parks in candidate pool: 252


/tmp/ipykernel_14634/1171462768.py:18: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  | alerts_df["combined_text_raw"].str.lower().str.contains(



## 5. Build rules for finding candidates for human review.

Rules below used to find records for review.

Each alert receives a score for every proposed class. The highest-scoring class becomes its suggested_sampling_stratum.


In [ ]:

KEYWORD_PATTERNS = {
    "trail_or_area_access": [
        r"\btrail(s)?\b", r"\barea closure\b", r"\bclosed to (the )?public\b",
        r"\bbeach access\b", r"\bcave(s)?\b", r"\bboardwalk\b",
        r"\bclimbing\b", r"\bbackcountry\b", r"\bpedestrian access\b",
        r"\bpartial(ly)? closed\b", r"\baccess restriction\b",
    ],
    "road_parking_transportation": [
        r"\broad(s)?\b", r"\bparking\b", r"\btraffic\b", r"\bvehicle(s)?\b",
        r"\bshuttle\b", r"\bferry\b", r"\bdetour\b", r"\blane(s)?\b",
        r"\bbridge\b", r"\bhighway\b", r"\bramp\b", r"\btransportation\b",
    ],
    "weather_fire_environmental_hazard": [
        r"\bfire\b", r"\bwildfire\b", r"\bsmoke\b", r"\bheat\b",
        r"\bflood", r"\bhigh water\b", r"\bwater quality\b", r"\bbacteria\b",
        r"\balgae\b", r"\bavalanche\b", r"\brockfall\b", r"\bunstable slope\b",
        r"\bstorm\b", r"\bhurricane\b", r"\brip current\b", r"\bsurf\b",
        r"\bice\b", r"\bsnow\b", r"\bdebris\b", r"\bhazardous conditions\b",
    ],
    "wildlife_hazard_or_restriction": [
        r"\bbear(s)?\b", r"\belk\b", r"\bbison\b", r"\bwildlife\b",
        r"\bnesting\b", r"\bbat(s)?\b", r"\bbird(s)?\b", r"\bplover(s)?\b",
        r"\btick(s)?\b", r"\bsnake(s)?\b", r"\bshark(s)?\b",
        r"\banimal(s)?\b", r"\bcalves\b", r"\bdistemper\b",
    ],
    "facility_water_campground_service": [
        r"\bvisitor center\b", r"\bcampground\b", r"\brestroom(s)?\b",
        r"\bdrinking water\b", r"\bwater unavailable\b", r"\bdock\b",
        r"\blaunch ramp\b", r"\bphone service\b", r"\bphone lines?\b",
        r"\belectricity\b", r"\bpower outage\b", r"\bfuel\b",
        r"\btour(s)?\b", r"\bpass(es)?\b", r"\bbookstore\b",
        r"\bheadquarters\b", r"\bmuseum\b", r"\btheater\b",
        r"\bfacilit(y|ies)\b", r"\bservices?\b", r"\blodging\b",
    ],
    "construction_maintenance_general": [
        r"\bconstruction\b", r"\bmaintenance\b", r"\brepair(s)?\b",
        r"\brestoration\b", r"\brehabilitation\b", r"\bpreservation\b",
        r"\bpermit(s)?\b", r"\bpolicy\b", r"\bscam\b", r"\btheft\b",
        r"\bbreak-?in(s)?\b", r"\bpayment\b", r"\bcashless\b",
        r"\bhours\b", r"\binformation\b", r"\breminder\b",
    ],
}

def keyword_score(text, patterns):
    return sum(bool(re.search(pattern, text, flags=re.I)) for pattern in patterns)

for label, patterns in KEYWORD_PATTERNS.items():
    candidate_unique[f"score__{label}"] = candidate_unique["combined_text_raw"].map(
        lambda text: keyword_score(text, patterns)
    )

score_columns = [f"score__{label}" for label in PRIMARY_LABELS]
candidate_unique["max_heuristic_score"] = candidate_unique[score_columns].max(axis=1)

candidate_unique["suggested_sampling_stratum"] = (
    candidate_unique[score_columns]
    .idxmax(axis=1)
    .str.replace("score__", "", regex=False)
)

# When all scores are zero, send the alert to the broad general-information review stratum.
candidate_unique.loc[
    candidate_unique["max_heuristic_score"].eq(0),
    "suggested_sampling_stratum"
] = "construction_maintenance_general"

display(
    candidate_unique[
        ["title", "category", "suggested_sampling_stratum", "max_heuristic_score"]
    ].head(15)
)


,title,category,suggested_sampling_stratum,max_heuristic_score
293,Several Trails Close For Peregrine Falcon Nest...,Park Closure,trail_or_area_access,1
52,Park Loop Road Detour,Park Closure,road_parking_transportation,2
159,Bag Policy,Park Closure,construction_maintenance_general,1
398,Intermittent Visitor Center Phone Issues,Information,facility_water_campground_service,1
136,Group Reservations,Information,facility_water_campground_service,1
226,Phone Messaging System Errors,Information,construction_maintenance_general,2
172,2026 Tour Season,Information,facility_water_campground_service,2
602,Limited Cell Phone Coverage,Information,facility_water_campground_service,1
321,"Hospital wing, dining hall, shower room closed...",Information,construction_maintenance_general,2
217,"Road Construction Work to begin May 4-Oct 24, ...",Information,construction_maintenance_general,2



## 6. Select approximately 180 expansion records

Greedy selection gives priority to the largest current class deficits, strong heuristic matches, parks not already heavily represented,lower duplicate-group sizes, and text-length diversity.

To preserve geographic diversity, the default maximum is three new candidates from any one park.


In [ ]:

EXPANSION_N = TARGET_TOTAL - len(gold_120_df)
MAX_NEW_PER_PARK = 3

existing_park_counts = gold_120_df["parkCode"].fillna("MISSING").value_counts().to_dict()
selected_park_counts = Counter()
selected_ids = set()
selected_rows = []

remaining_need = target_deficits.astype(int).to_dict()

# Rank within each suggested stratum.
ranked_candidates = candidate_unique.copy()
ranked_candidates["park_existing_count"] = (
    ranked_candidates["parkCode"].fillna("MISSING").map(existing_park_counts).fillna(0)
)
ranked_candidates["text_length_distance"] = (
    ranked_candidates["combined_text_raw"].str.split().str.len() - 45
).abs()

ranked_candidates = ranked_candidates.sort_values(
    [
        "max_heuristic_score",
        "park_existing_count",
        "duplicate_text_group_size_global",
        "text_length_distance",
    ],
    ascending=[False, True, True, True],
)

def can_select(row):
    park = row["parkCode"] if pd.notna(row["parkCode"]) else "MISSING"
    return (
        row["id"] not in selected_ids
        and selected_park_counts[park] < MAX_NEW_PER_PARK
    )

# First pass: fill each planned class deficit.
for label in sorted(
    PRIMARY_LABELS,
    key=lambda value: remaining_need[value],
    reverse=True
):
    needed = remaining_need[label]
    stratum_rows = ranked_candidates.loc[
        ranked_candidates["suggested_sampling_stratum"].eq(label)
    ]

    for _, row in stratum_rows.iterrows():
        if needed <= 0:
            break
        if can_select(row):
            selected_rows.append(row)
            selected_ids.add(row["id"])
            park = row["parkCode"] if pd.notna(row["parkCode"]) else "MISSING"
            selected_park_counts[park] += 1
            needed -= 1

    remaining_need[label] = needed

# Second pass: fill any shortfall with high-value unselected candidates.
if len(selected_rows) < EXPANSION_N:
    for _, row in ranked_candidates.iterrows():
        if len(selected_rows) >= EXPANSION_N:
            break
        if can_select(row):
            selected_rows.append(row)
            selected_ids.add(row["id"])
            park = row["parkCode"] if pd.notna(row["parkCode"]) else "MISSING"
            selected_park_counts[park] += 1

expansion_df = pd.DataFrame(selected_rows).copy()
expansion_df.insert(
    0,
    "review_id",
    [f"E{i:03d}" for i in range(1, len(expansion_df) + 1)]
)
expansion_df["source_stage"] = "expansion_candidate"
expansion_df["primary_label"] = ""
expansion_df["secondary_label"] = ""
expansion_df["impact_label"] = ""
expansion_df["uncertain"] = 0
expansion_df["uncertainty_reason"] = ""
expansion_df["annotation_notes"] = ""
expansion_df["review_status"] = "pending"

print("Expansion records selected:", len(expansion_df))
print("Unique parks selected:", expansion_df["parkCode"].nunique())
print("Maximum new records from one park:", expansion_df["parkCode"].value_counts().max())

print("\nSuggested sampling strata:")
display(
    expansion_df["suggested_sampling_stratum"]
    .value_counts()
    .reindex(PRIMARY_LABELS, fill_value=0)
    .to_frame("count")
)

print("\nUnfilled planned deficits after the first pass:")
display(pd.Series(remaining_need, name="unfilled_deficit").to_frame())


Expansion records selected: 180
Unique parks selected: 134
Maximum new records from one park: 3

Suggested sampling strata:


,count
suggested_sampling_stratum,
trail_or_area_access,23
road_parking_transportation,47
weather_fire_environmental_hazard,29
wildlife_hazard_or_restriction,11
facility_water_campground_service,27
construction_maintenance_general,43



Unfilled planned deficits after the first pass:


,unfilled_deficit
trail_or_area_access,0
road_parking_transportation,0
weather_fire_environmental_hazard,1
wildlife_hazard_or_restriction,31
facility_water_campground_service,0
construction_maintenance_general,0



## 7. Export the expansion annotation workbook



In [ ]:

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.utils import get_column_letter

preferred_output_dirs = [
    Path("/content/nps_project_outputs"),
    Path("/mnt/data/nps_project_outputs"),
    Path("nps_project_outputs"),
]

output_dir = None
for candidate_dir in preferred_output_dirs:
    try:
        candidate_dir.mkdir(parents=True, exist_ok=True)
        output_dir = candidate_dir
        break
    except (PermissionError, OSError):
        continue

if output_dir is None:
    raise PermissionError("Could not create an output directory.")

print("Output directory:", output_dir.resolve())

expansion_columns = [
    "review_id", "id", "parkCode", "fullName", "category",
    "title", "description", "combined_text_raw",
    "combined_word_count", "duplicate_text_group_size_global",
    "suggested_sampling_stratum", "max_heuristic_score",
    "primary_label", "secondary_label", "impact_label",
    "uncertain", "uncertainty_reason", "annotation_notes",
    "review_status", "source_stage"
]

# Keep only available columns in the intended order.
expansion_columns = [c for c in expansion_columns if c in expansion_df.columns]
expansion_export = expansion_df[expansion_columns].copy()

EXPANSION_WORKBOOK_PATH = output_dir / "nps_gold_expansion_annotation_180.xlsx"

with pd.ExcelWriter(EXPANSION_WORKBOOK_PATH, engine="openpyxl") as writer:
    expansion_export.to_excel(writer, index=False, sheet_name="Expansion_Annotations")

    taxonomy_guide = pd.DataFrame([
        {
            "primary_label": "trail_or_area_access",
            "use_when": "A trail, beach, cave, plaza, zone, or visitor area is closed, restricted, or limited.",
            "boundary_rule": "Trail closed for repairs: trail/area primary; maintenance secondary."
        },
        {
            "primary_label": "road_parking_transportation",
            "use_when": "Road access, parking, shuttles, ferries, vehicles, traffic, bridges, or detours are central.",
            "boundary_rule": "Road closed by storm: road primary; environmental hazard secondary."
        },
        {
            "primary_label": "weather_fire_environmental_hazard",
            "use_when": "Fire, heat, smoke, flooding, water quality, rockfall, unstable terrain, surf, or natural hazards drive the alert.",
            "boundary_rule": "Trail closed by rockfall: environmental hazard primary; trail secondary."
        },
        {
            "primary_label": "wildlife_hazard_or_restriction",
            "use_when": "Animals, nesting, wildlife disease, or wildlife protection are the main cause of risk or restriction.",
            "boundary_rule": "Campground closed for bear activity: wildlife primary; facility secondary."
        },
        {
            "primary_label": "facility_water_campground_service",
            "use_when": "A visitor center, campground, dock, ramp, restroom, tour, pass, phone, fuel, or service is changed or unavailable.",
            "boundary_rule": "Dock closed for construction: facility primary; maintenance secondary."
        },
        {
            "primary_label": "construction_maintenance_general",
            "use_when": "Administrative, informational, permit, theft/scam, policy, or minor maintenance notice with no more specific consequence.",
            "boundary_rule": "Use only when no more specific traveler-facing class applies."
        },
    ])
    taxonomy_guide.to_excel(writer, index=False, sheet_name="Taxonomy_Guide")

    impact_guide = pd.DataFrame([
        {
            "impact_label": "trip_changing",
            "rule": "A major itinerary or central activity change is likely."
        },
        {
            "impact_label": "requires_preparation",
            "rule": "The trip can continue with precautions, delays, restrictions, equipment, or adjustments."
        },
        {
            "impact_label": "informational_minimal_impact",
            "rule": "Useful information with little immediate disruption."
        },
    ])
    impact_guide.to_excel(writer, index=False, sheet_name="Impact_Guide")

# Add formatting and dropdown validation.
workbook = __import__("openpyxl").load_workbook(EXPANSION_WORKBOOK_PATH)
worksheet = workbook["Expansion_Annotations"]
worksheet.freeze_panes = "A2"
worksheet.auto_filter.ref = worksheet.dimensions

header_fill = PatternFill("solid", fgColor="1F4E78")
header_font = Font(color="FFFFFF", bold=True)

for cell in worksheet[1]:
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

column_index = {cell.value: cell.column for cell in worksheet[1]}

for name, width in {
    "review_id": 10, "id": 18, "parkCode": 10, "fullName": 30,
    "category": 16, "title": 38, "description": 70, "combined_text_raw": 70,
    "combined_word_count": 14, "duplicate_text_group_size_global": 14,
    "suggested_sampling_stratum": 32, "max_heuristic_score": 14,
    "primary_label": 32, "secondary_label": 32, "impact_label": 27,
    "uncertain": 11, "uncertainty_reason": 26, "annotation_notes": 45,
    "review_status": 14, "source_stage": 18,
}.items():
    if name in column_index:
        worksheet.column_dimensions[
            get_column_letter(column_index[name])
        ].width = width

for row in worksheet.iter_rows(min_row=2):
    for cell in row:
        cell.alignment = Alignment(vertical="top", wrap_text=True)

primary_list = '"' + ",".join(PRIMARY_LABELS) + '"'
impact_list = '"' + ",".join(IMPACT_LABELS) + '"'

primary_validation = DataValidation(type="list", formula1=primary_list, allow_blank=True)
secondary_validation = DataValidation(type="list", formula1=primary_list, allow_blank=True)
impact_validation = DataValidation(type="list", formula1=impact_list, allow_blank=True)
uncertain_validation = DataValidation(type="list", formula1='"0,1"', allow_blank=False)
status_validation = DataValidation(type="list", formula1='"pending,reviewed"', allow_blank=False)

worksheet.add_data_validation(primary_validation)
worksheet.add_data_validation(secondary_validation)
worksheet.add_data_validation(impact_validation)
worksheet.add_data_validation(uncertain_validation)
worksheet.add_data_validation(status_validation)

last_row = worksheet.max_row
primary_validation.add(
    f"{get_column_letter(column_index['primary_label'])}2:"
    f"{get_column_letter(column_index['primary_label'])}{last_row}"
)
secondary_validation.add(
    f"{get_column_letter(column_index['secondary_label'])}2:"
    f"{get_column_letter(column_index['secondary_label'])}{last_row}"
)
impact_validation.add(
    f"{get_column_letter(column_index['impact_label'])}2:"
    f"{get_column_letter(column_index['impact_label'])}{last_row}"
)
uncertain_validation.add(
    f"{get_column_letter(column_index['uncertain'])}2:"
    f"{get_column_letter(column_index['uncertain'])}{last_row}"
)
status_validation.add(
    f"{get_column_letter(column_index['review_status'])}2:"
    f"{get_column_letter(column_index['review_status'])}{last_row}"
)

workbook.save(EXPANSION_WORKBOOK_PATH)

print("Saved expansion annotation workbook:")
print(EXPANSION_WORKBOOK_PATH)


Output directory: /content/nps_project_outputs
Saved expansion annotation workbook:
/content/nps_project_outputs/nps_gold_expansion_annotation_180.xlsx



# Run after completing expansion annotations

Upload the completed expansion workbook. The remaining cells validate the expanded gold set and create the two evaluation settings.


In [ ]:

COMPLETED_EXPANSION_FILENAME = "nps_gold_expansion_annotation_180_COMPLETED.xlsx"

completed_expansion_path = None
for directory in SEARCH_DIRS + [output_dir]:
    candidate = directory / COMPLETED_EXPANSION_FILENAME
    if candidate.exists():
        completed_expansion_path = candidate
        break

if completed_expansion_path is None:
    print(
        "Completed expansion workbook not found yet. "
        "Finish review, upload the completed file, and rerun this cell."
    )
else:
    print("Completed expansion file:", completed_expansion_path)


Completed expansion file: /content/nps_gold_expansion_annotation_180_COMPLETED.xlsx


## 8. Load and validate the expanded gold-standard dataset

In [ ]:

if completed_expansion_path is None:
    raise FileNotFoundError(
        "Complete and upload the expansion annotation workbook before continuing."
    )

expansion_labeled_df = pd.read_excel(
    completed_expansion_path,
    sheet_name="Expansion_Annotations"
)

required_annotation_fields = [
    "primary_label", "impact_label", "uncertain", "review_status"
]

for column in required_annotation_fields:
    if column not in expansion_labeled_df.columns:
        raise ValueError(f"Missing annotation field: {column}")

expansion_labeled_df["primary_label"] = (
    expansion_labeled_df["primary_label"].fillna("").astype(str).str.strip()
)
expansion_labeled_df["impact_label"] = (
    expansion_labeled_df["impact_label"].fillna("").astype(str).str.strip()
)

incomplete_mask = (
    expansion_labeled_df["primary_label"].eq("")
    | expansion_labeled_df["impact_label"].eq("")
    | ~expansion_labeled_df["review_status"].eq("reviewed")
)

print("Expansion rows:", len(expansion_labeled_df))
print("Incomplete expansion annotations:", int(incomplete_mask.sum()))

if incomplete_mask.any():
    display(
        expansion_labeled_df.loc[
            incomplete_mask,
            ["review_id", "title", "primary_label", "impact_label", "review_status"]
        ].head(25)
    )
    raise ValueError("Finish all expansion annotations before creating splits.")

invalid_primary = set(expansion_labeled_df["primary_label"]) - set(PRIMARY_LABELS)
invalid_impact = set(expansion_labeled_df["impact_label"]) - set(IMPACT_LABELS)

if invalid_primary:
    raise ValueError(f"Invalid primary labels: {sorted(invalid_primary)}")
if invalid_impact:
    raise ValueError(f"Invalid impact labels: {sorted(invalid_impact)}")

gold_120_df["source_stage"] = "completed_pilot"
expanded_gold_df = pd.concat(
    [gold_120_df, expansion_labeled_df],
    ignore_index=True,
    sort=False
)

print("Expanded gold dataset:", expanded_gold_df.shape)
display(expanded_gold_df["primary_label"].value_counts().to_frame("count"))


Expansion rows: 180
Incomplete expansion annotations: 0
Expanded gold dataset: (300, 24)


,count
primary_label,
road_parking_transportation,86
facility_water_campground_service,84
trail_or_area_access,53
weather_fire_environmental_hazard,44
wildlife_hazard_or_restriction,20
construction_maintenance_general,13



## 9. Deduplicate exact text before splitting


In [ ]:

expanded_gold_df["normalized_text_modeling"] = expanded_gold_df.get(
    "normalized_text",
    expanded_gold_df["combined_text_raw"].map(normalize_text)
)

expanded_gold_df["normalized_text_modeling"] = (
    expanded_gold_df["normalized_text_modeling"]
    .fillna(expanded_gold_df["combined_text_raw"].map(normalize_text))
    .astype(str)
)

duplicate_audit = (
    expanded_gold_df
    .groupby("normalized_text_modeling", dropna=False)
    .agg(
        records=("id", "size"),
        parks=("parkCode", "nunique"),
        labels=("primary_label", lambda values: sorted(set(values))),
        ids=("id", list),
    )
    .reset_index()
)

duplicate_groups = duplicate_audit.loc[duplicate_audit["records"].gt(1)].copy()
label_conflicts = duplicate_groups.loc[
    duplicate_groups["labels"].map(len).gt(1)
].copy()

print("Exact duplicate-text groups:", len(duplicate_groups))
print("Duplicate groups with conflicting human labels:", len(label_conflicts))

if len(label_conflicts):
    display(label_conflicts.head(20))
    raise ValueError(
        "Resolve conflicting labels among exact duplicate texts before splitting."
    )

modeling_df = (
    expanded_gold_df
    .sort_values(["normalized_text_modeling", "id"])
    .drop_duplicates("normalized_text_modeling", keep="first")
    .reset_index(drop=True)
)

print("Gold records before exact-text deduplication:", len(expanded_gold_df))
print("Modeling records after deduplication:", len(modeling_df))
print("Unique parks:", modeling_df["parkCode"].nunique())
display(modeling_df["primary_label"].value_counts().to_frame("count"))


Exact duplicate-text groups: 0
Duplicate groups with conflicting human labels: 0
Gold records before exact-text deduplication: 300
Modeling records after deduplication: 300
Unique parks: 199


,count
primary_label,
road_parking_transportation,86
facility_water_campground_service,84
trail_or_area_access,53
weather_fire_environmental_hazard,44
wildlife_hazard_or_restriction,20
construction_maintenance_general,13



## 10. Create primary held-out-park split

This search evaluates many grouped split candidates and chooses the one that best balances:

- class proportions,
- class presence in validation and test,
- complete park separation.

No parkCode may appear in more than one partition.


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

def class_distribution(frame):
    return (
        frame["primary_label"]
        .value_counts(normalize=True)
        .reindex(PRIMARY_LABELS, fill_value=0)
    )

overall_distribution = class_distribution(modeling_df)
n_total = len(modeling_df)

def heldout_split_score(train_idx, val_idx, test_idx):
    split_indices = {
        "train": np.array(train_idx),
        "validation": np.array(val_idx),
        "test": np.array(test_idx),
    }

    target_sizes = {
        "train": 0.70,
        "validation": 0.15,
        "test": 0.15,
    }

    score = 0.0

    for split_name, indices in split_indices.items():
        frame = modeling_df.iloc[indices]

        # Keep split sizes close to 70/15/15.
        actual_size = len(frame) / n_total
        score += 6.0 * abs(
            actual_size - target_sizes[split_name]
        )

        # Keep class proportions reasonably close to the full dataset.
        distribution_gap = (
            class_distribution(frame)
            - overall_distribution
        ).abs().mean()

        score += 3.0 * distribution_gap

        counts = (
            frame["primary_label"]
            .value_counts()
            .reindex(PRIMARY_LABELS, fill_value=0)
        )

        if split_name in ["validation", "test"]:
            # Strongly penalize any class with fewer than two examples.
            too_small = (counts < 2).sum()
            score += 20.0 * too_small

            # Extra protection for the rare general/minor class.
            general_count = counts[
                "construction_maintenance_general"
            ]

            if general_count < 2:
                score += 30.0

        else:
            # Preserve enough training examples for each class.
            too_small = (counts < 5).sum()
            score += 20.0 * too_small

    return score


best_grouped = None

# Search more possible grouped splits.
for seed in range(3000):
    outer = GroupShuffleSplit(
        n_splits=1,
        test_size=0.15,
        random_state=seed,
    )

    train_val_idx, test_idx = next(
        outer.split(
            modeling_df,
            y=modeling_df["primary_label"],
            groups=modeling_df["parkCode"],
        )
    )

    train_val = modeling_df.iloc[train_val_idx]

    # Validation should be 15% of the full dataset.
    relative_val_size = 0.15 / 0.85

    inner = GroupShuffleSplit(
        n_splits=1,
        test_size=relative_val_size,
        random_state=seed + 10_000,
    )

    train_rel_idx, val_rel_idx = next(
        inner.split(
            train_val,
            y=train_val["primary_label"],
            groups=train_val["parkCode"],
        )
    )

    train_idx = train_val_idx[train_rel_idx]
    val_idx = train_val_idx[val_rel_idx]

    score = heldout_split_score(
        train_idx,
        val_idx,
        test_idx,
    )

    if (
        best_grouped is None
        or score < best_grouped["score"]
    ):
        best_grouped = {
            "score": score,
            "seed": seed,
            "train_idx": train_idx,
            "validation_idx": val_idx,
            "test_idx": test_idx,
        }


heldout_split = pd.Series(
    index=modeling_df.index,
    dtype="object",
)

heldout_split.iloc[
    best_grouped["train_idx"]
] = "train"

heldout_split.iloc[
    best_grouped["validation_idx"]
] = "validation"

heldout_split.iloc[
    best_grouped["test_idx"]
] = "test"

modeling_df["heldout_park_split"] = heldout_split

print(
    "Selected grouped-split seed:",
    best_grouped["seed"],
)

print(
    "Grouped-split optimization score:",
    round(best_grouped["score"], 4),
)

# Confirm that each park belongs to only one partition.
park_split_counts = (
    modeling_df
    .groupby("parkCode")["heldout_park_split"]
    .nunique()
)

assert (
    park_split_counts.max() == 1
), "A park appears in multiple held-out partitions."

heldout_class_counts = (
    pd.crosstab(
        modeling_df["primary_label"],
        modeling_df["heldout_park_split"],
    )
    .reindex(
        PRIMARY_LABELS,
        fill_value=0,
    )
    .reindex(
        columns=["train", "validation", "test"],
        fill_value=0,
    )
)

display(heldout_class_counts)

print("\nRecords per split:")
display(
    modeling_df["heldout_park_split"]
    .value_counts()
    .reindex(["train", "validation", "test"])
    .to_frame("count")
)

print("\nUnique parks per split:")
display(
    modeling_df
    .groupby("heldout_park_split")["parkCode"]
    .nunique()
    .reindex(["train", "validation", "test"])
    .to_frame("unique_parks")
)

# Final safeguards.
assert (
    heldout_class_counts["validation"] >= 2
).all(), "At least one validation class has fewer than two records."

assert (
    heldout_class_counts["test"] >= 2
).all(), "At least one test class has fewer than two records."

print(
    "\nAll classes have at least two records "
    "in both validation and test."
)


Selected grouped-split seed: 1552
Grouped-split optimization score: 0.1831


heldout_park_split,train,validation,test
primary_label,,,
trail_or_area_access,36,9,8
road_parking_transportation,62,10,14
weather_fire_environmental_hazard,31,6,7
wildlife_hazard_or_restriction,16,2,2
facility_water_campground_service,57,15,12
construction_maintenance_general,9,2,2



Records per split:


,count
heldout_park_split,
train,211
validation,44
test,45



Unique parks per split:


,unique_parks
heldout_park_split,
train,139
validation,30
test,30



All classes have at least two records in both validation and test.



## 11. Create the secondary mixed-park split

This split is stratified by class but does not keep parks separate. It is expected to be easier and will be reported after the held-out-park result.


In [ ]:

from sklearn.model_selection import train_test_split

indices = np.arange(len(modeling_df))

train_val_idx, mixed_test_idx = train_test_split(
    indices,
    test_size=0.15,
    stratify=modeling_df["primary_label"],
    random_state=RANDOM_STATE,
)

relative_val_size = 0.15 / 0.85

mixed_train_idx, mixed_val_idx = train_test_split(
    train_val_idx,
    test_size=relative_val_size,
    stratify=modeling_df.iloc[train_val_idx]["primary_label"],
    random_state=RANDOM_STATE,
)

mixed_split = pd.Series(index=modeling_df.index, dtype="object")
mixed_split.iloc[mixed_train_idx] = "train"
mixed_split.iloc[mixed_val_idx] = "validation"
mixed_split.iloc[mixed_test_idx] = "test"

modeling_df["mixed_park_split"] = mixed_split

display(
    pd.crosstab(
        modeling_df["primary_label"],
        modeling_df["mixed_park_split"]
    ).reindex(PRIMARY_LABELS, fill_value=0)
)

print("\nRecords per split:")
display(modeling_df["mixed_park_split"].value_counts().to_frame("count"))


mixed_park_split,test,train,validation
primary_label,,,
trail_or_area_access,8,37,8
road_parking_transportation,13,60,13
weather_fire_environmental_hazard,7,31,6
wildlife_hazard_or_restriction,3,14,3
facility_water_campground_service,12,59,13
construction_maintenance_general,2,9,2



Records per split:


,count
mixed_park_split,
train,210
test,45
validation,45



## 12. Create split diagnostics

Held-out-park results first, Mixed-park results second, Direct comparison of the generalization gap


In [ ]:

def split_diagnostics(frame, split_column):
    count_table = (
        pd.crosstab(frame["primary_label"], frame[split_column])
        .reindex(PRIMARY_LABELS, fill_value=0)
        .reindex(columns=["train", "validation", "test"], fill_value=0)
    )

    percent_table = count_table.div(
        count_table.sum(axis=0), axis=1
    ).round(3)

    size_table = (
        frame[split_column]
        .value_counts()
        .reindex(["train", "validation", "test"])
        .rename("records")
        .to_frame()
    )
    size_table["percent"] = (
        size_table["records"] / len(frame) * 100
    ).round(1)

    return count_table, percent_table, size_table

heldout_counts, heldout_pct, heldout_sizes = split_diagnostics(
    modeling_df, "heldout_park_split"
)
mixed_counts, mixed_pct, mixed_sizes = split_diagnostics(
    modeling_df, "mixed_park_split"
)

print("PRIMARY SETTING: HELD-OUT PARKS")
display(heldout_sizes)
display(heldout_counts)
display(heldout_pct)

print("\nSECONDARY SETTING: MIXED PARKS")
display(mixed_sizes)
display(mixed_counts)
display(mixed_pct)


PRIMARY SETTING: HELD-OUT PARKS


,records,percent
heldout_park_split,,
train,211,70.3
validation,44,14.7
test,45,15.0


heldout_park_split,train,validation,test
primary_label,,,
trail_or_area_access,36,9,8
road_parking_transportation,62,10,14
weather_fire_environmental_hazard,31,6,7
wildlife_hazard_or_restriction,16,2,2
facility_water_campground_service,57,15,12
construction_maintenance_general,9,2,2


heldout_park_split,train,validation,test
primary_label,,,
trail_or_area_access,0.171,0.205,0.178
road_parking_transportation,0.294,0.227,0.311
weather_fire_environmental_hazard,0.147,0.136,0.156
wildlife_hazard_or_restriction,0.076,0.045,0.044
facility_water_campground_service,0.270,0.341,0.267
construction_maintenance_general,0.043,0.045,0.044



SECONDARY SETTING: MIXED PARKS


,records,percent
mixed_park_split,,
train,210,70.0
validation,45,15.0
test,45,15.0


mixed_park_split,train,validation,test
primary_label,,,
trail_or_area_access,37,8,8
road_parking_transportation,60,13,13
weather_fire_environmental_hazard,31,6,7
wildlife_hazard_or_restriction,14,3,3
facility_water_campground_service,59,13,12
construction_maintenance_general,9,2,2


mixed_park_split,train,validation,test
primary_label,,,
trail_or_area_access,0.176,0.178,0.178
road_parking_transportation,0.286,0.289,0.289
weather_fire_environmental_hazard,0.148,0.133,0.156
wildlife_hazard_or_restriction,0.067,0.067,0.067
facility_water_campground_service,0.281,0.289,0.267
construction_maintenance_general,0.043,0.044,0.044


## 13. Export the frozen modeling dataset and split files

In [ ]:

MODEL_COLUMNS = [
    "id", "parkCode", "fullName", "category", "title", "description",
    "combined_text_raw", "normalized_text_modeling",
    "primary_label", "secondary_label", "impact_label",
    "heldout_park_split", "mixed_park_split", "source_stage",
]

MODEL_COLUMNS = [column for column in MODEL_COLUMNS if column in modeling_df.columns]

modeling_path = output_dir / "nps_gold_modeling_dataset_with_splits.csv"
modeling_df[MODEL_COLUMNS].to_csv(modeling_path, index=False)

heldout_dir = output_dir / "heldout_park"
mixed_dir = output_dir / "mixed_park"
heldout_dir.mkdir(exist_ok=True)
mixed_dir.mkdir(exist_ok=True)

for split_name in ["train", "validation", "test"]:
    modeling_df.loc[
        modeling_df["heldout_park_split"].eq(split_name),
        MODEL_COLUMNS
    ].to_csv(
        heldout_dir / f"{split_name}.csv",
        index=False
    )

    modeling_df.loc[
        modeling_df["mixed_park_split"].eq(split_name),
        MODEL_COLUMNS
    ].to_csv(
        mixed_dir / f"{split_name}.csv",
        index=False
    )

duplicate_audit.to_csv(
    output_dir / "nps_exact_duplicate_audit.csv",
    index=False
)

with pd.ExcelWriter(
    output_dir / "nps_split_diagnostics.xlsx",
    engine="openpyxl"
) as writer:
    heldout_sizes.to_excel(writer, sheet_name="Heldout_Sizes")
    heldout_counts.to_excel(writer, sheet_name="Heldout_Class_Counts")
    heldout_pct.to_excel(writer, sheet_name="Heldout_Class_Proportions")
    mixed_sizes.to_excel(writer, sheet_name="Mixed_Sizes")
    mixed_counts.to_excel(writer, sheet_name="Mixed_Class_Counts")
    mixed_pct.to_excel(writer, sheet_name="Mixed_Class_Proportions")

print("Saved modeling dataset:", modeling_path)
print("Saved held-out-park split files:", heldout_dir)
print("Saved mixed-park split files:", mixed_dir)
print("Saved diagnostics:", output_dir / "nps_split_diagnostics.xlsx")


Saved modeling dataset: /content/nps_project_outputs/nps_gold_modeling_dataset_with_splits.csv
Saved held-out-park split files: /content/nps_project_outputs/heldout_park
Saved mixed-park split files: /content/nps_project_outputs/mixed_park
Saved diagnostics: /content/nps_project_outputs/nps_split_diagnostics.xlsx
